# Qwen3-TTS Voice Cloning

Model: `Qwen3-TTS-12Hz-1.7B-Base` | Reference: `audio/laxmikant_en.wav`

| Parameter | Effect |
|---|---|
| `instruct` | natural-language delivery instruction |
| `temperature` | 0.7-1.0, higher = more expressive |
| `top_p` | nucleus cutoff |
| `repetition_penalty` | reduces phoneme artifacts |

## 1. Setup

In [1]:
import warnings
import torch, soundfile as sf
import IPython.display as ipd
from pathlib import Path
from qwen_tts import Qwen3TTSModel

warnings.filterwarnings("ignore")

# ── config ────────────────────────────────────────────────────────────────
REF_AUDIO  = Path("audio/ref_en.wav")
REF_TEXT   = Path("audio/ref_en.txt")
OUTPUT_DIR = Path("output")
# ──────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)

SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


## 2. Load Model

In [2]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=torch.bfloat16
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

## 3. Voice Prompt

Edit `REF_TEXT` to match what is spoken in your reference audio. Accuracy improves clone quality.

In [3]:
ref_text = REF_TEXT.read_text(encoding="utf-8").strip()
voice_prompt = model.create_voice_clone_prompt(ref_audio=str(REF_AUDIO), ref_text=ref_text)

## 4. Expressive Presets

In [4]:
BASE = dict(
    language="English",
    voice_clone_prompt=voice_prompt,
    temperature=0.9,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,
    max_new_tokens=4096,
)

PRESETS = {
    "neutral":       {"instruct": "Speak clearly and naturally at a moderate pace."},
    "excited":       {"instruct": "High energy and enthusiasm, slightly faster pace.", "temperature": 0.95},
    "authoritative": {"instruct": "Calm, confident tone like a senior engineer presenting to stakeholders.", "temperature": 0.75, "top_p": 0.85},
    "thoughtful":    {"instruct": "Curious and thoughtful, pausing slightly before key ideas.", "temperature": 0.88},
    "storytelling":  {"instruct": "Compelling storytelling with natural rises and falls in pitch and pace.", "temperature": 0.95},
}

## 5. Helper

In [5]:
def synthesize(text: str, preset: str = "neutral", filename: str | None = None):
    params = BASE.copy()
    params.update(PRESETS[preset])
    wavs, sr = model.generate_voice_clone(text=text, **params)
    out = OUTPUT_DIR / (filename or f"{preset}.wav")
    sf.write(str(out), wavs[0], sr)
    ipd.display(ipd.Audio(wavs[0], rate=sr))

## 6. Sanity Check

In [6]:
synthesize(
    "The voice cloning system is working correctly. This is a quick sanity check.",
    preset="neutral",
    filename="01_sanity.wav",
)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 7. Presets Showcase

In [7]:
sample = (
    "Large language models are reshaping how we build software. "
    "They reason across context, generate code, and learn from feedback."
)

for preset in PRESETS:
    print(preset)
    synthesize(sample, preset=preset, filename=f"02_{preset}.wav")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


neutral


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


excited


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


authoritative


KeyboardInterrupt: 

## 8. Custom Instruct

In [8]:
params = BASE.copy()
params["instruct"]    = "Quiet conviction, reflecting on hard-won experience. Slower pace, measured pauses."
params["temperature"] = 0.82
params["top_p"]       = 0.87

text = (
    "Building something from scratch is never easy. "
    "Every failure teaches you something no book ever could."
)

wavs, sr = model.generate_voice_clone(text=text, **params)
sf.write(str(OUTPUT_DIR / "03_custom.wav"), wavs[0], sr)
ipd.display(ipd.Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 9. X-Vector Mode (no transcript needed)

In [9]:
x_prompt = model.create_voice_clone_prompt(ref_audio=str(REF_AUDIO), x_vector_only_mode=True)

wavs, sr = model.generate_voice_clone(
    text="X-vector mode skips the transcript and uses a speaker embedding only.",
    language="English",
    voice_clone_prompt=x_prompt,
    instruct="Speak naturally and clearly.",
    temperature=0.88,
    top_p=0.9,
    repetition_penalty=1.1,
)
sf.write(str(OUTPUT_DIR / "04_xvector.wav"), wavs[0], sr)
ipd.display(ipd.Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 10. Synthesize from Text Script

In [11]:
SCRIPT_PATH = Path("script/llm_story.txt")

text = SCRIPT_PATH.read_text(encoding="utf-8").strip()

wavs, sr = model.generate_voice_clone(
    text=text,
    language="English",
    voice_clone_prompt=voice_prompt,
    instruct="Warm, engaging storytelling. Natural and slow pace with gentle variation in pitch.",
    temperature=0.88,
    top_p=0.90,
    top_k=50,
    repetition_penalty=1.1,
    max_new_tokens=4096,
)

out_path = OUTPUT_DIR / f"{SCRIPT_PATH.stem}_cloned.wav"
sf.write(str(out_path), wavs[0], sr)
ipd.display(ipd.Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
